# 01 · 零售用户价值分析（RFM + Cohort + 聚类验证）> 数据集：**UCI Online Retail**（英国在线零售，2010-12-01 ~ 2011-12-09）> 目标：从 54 万条交易记录中识别高价值客户、量化留存结构，输出可落地的运营建议。## 本 Notebook 做了什么| 步骤 | 内容 | 产出 || --- | --- | --- || 1 | 数据读取与体检 | 字段字典、缺失率 || 2 | 数据清洗（5 条规则） | `retail_clean.csv`（约 39 万行） || 3 | EDA 探索 | 6 张图：月度趋势 / 时段热力 / TOP 商品 / TOP 国家 || 4 | RFM 用户分层 | `rfm.csv`（4,284 名客户 → 6 类人群） || 5 | KMeans 交叉验证 | 轮廓系数、ARI、覆盖率 || 6 | Cohort 同期群留存 | `retention.csv` / `retention_long.csv` || 7 | 复购率、客单价、国家 KPI | 汇总指标 || 8 | （可选）Apriori 购物篮 | 关联规则 |## 核心结论（先给结果）- **前 22.8% 的高价值客户贡献 69.5% 营收**，呈典型帕累托结构；- 高价值客户**流失率仅 1.1%**，因此运营重点应是「潜力客户升级」而非「流失唤醒」；- **新客次月留存仅 20.3%**，首购后 30 天是核心流失窗口；- 销售高度集中于**工作日 10-15 点、每年 Q4**（圣诞采购季）。

## 0. 环境与全局设置中文字体是必配项，否则所有图的标题会变成方框。

In [ ]:
# -*- coding: utf-8 -*-import osimport datetime as dtimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.preprocessing import StandardScalerfrom sklearn.cluster import KMeansfrom sklearn.metrics import silhouette_score, adjusted_rand_scorefrom scipy.stats import spearmanr# 中文字体（Windows）plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']plt.rcParams['axes.unicode_minus'] = Falsesns.set(font='SimHei', style='whitegrid')BASE = os.getcwd()                    # 项目根目录（在根目录启动 Jupyter 即可）os.makedirs('output', exist_ok=True)# ---- 自动定位原始数据：优先现成的 xlsx，没有就从仓库自带的 zip 解压 ----def _locate_source(base):    for p in (os.path.join(base, 'Online Retail.xlsx'),              os.path.join(base, 'data', 'Online Retail.xlsx'),              os.path.join(base, 'data', 'raw', 'Online Retail.xlsx')):        if os.path.exists(p):            return p    z = os.path.join(base, 'online+retail.zip')    if os.path.exists(z):        import zipfile        with zipfile.ZipFile(z) as zf:            name = next(n for n in zf.namelist() if n.lower().endswith(('.xlsx', '.xls')))            out = os.path.join(base, 'Online Retail.xlsx')            with zf.open(name) as src, open(out, 'wb') as dst:                dst.write(src.read())            print(f'已从压缩包解压：{name}')            return out    return NoneSRC = _locate_source(BASE)if SRC is None:    raise FileNotFoundError(        '未找到 Online Retail.xlsx。请从 UCI 下载后放到项目根目录或 data/ 下：'        'https://archive.ics.uci.edu/dataset/352/online+retail'    )print('工作目录：', BASE)print('数据文件：', SRC)

## 1. 数据读取与体检### 字段字典| 字段 | 含义 | 备注 || --- | --- | --- || `InvoiceNo` | 订单号 | 以 `C` 开头的是**退货/取消单** || `StockCode` | 商品编码 | || `Description` | 商品名称 | 有缺失与大小写不一致 || `Quantity` | 数量 | 负数 = 退货 || `InvoiceDate` | 下单时间 | 精确到分钟 || `UnitPrice` | 单价（英镑） | 存在 0 值 || `CustomerID` | 客户 ID | **缺失约 25%**，缺失行无法参与 RFM || `Country` | 国家 | 英国占绝大多数 |体检的三个目的：确认行数与类型、看缺失分布、锁定要清洗的脏数据规模。

In [ ]:
df = pd.read_excel(SRC, sheet_name='Online Retail',                   dtype={'CustomerID': str})   # 读成字符串，避免 17850.0 这种浮点污染print('形状：', df.shape)print('\n字段类型：'); print(df.dtypes)print('\n缺失率：'); print(df.isna().mean().round(4))print('\n前 5 行：'); print(df.head())print('\n数值摘要：'); print(df.describe().T)print('\n--- 脏数据规模 ---')print(f'非正数量占比      ：{(df["Quantity"] <= 0).mean():.2%}')print(f'退货单（C 开头）数：{df["InvoiceNo"].astype(str).str.startswith("C").sum():,}')print(f'单价为 0 的行数   ：{(df["UnitPrice"] == 0).sum():,}')print('\n国家 TOP10：'); print(df['Country'].value_counts().head(10))

## 2. 数据清洗五条规则，**顺序不能乱**（先剔除无客户 ID，再剔退货，最后才做异常值截断）：1. **剔除无 `CustomerID` 的行** —— RFM 必须能归属到用户，这部分约占 25%，只能丢；2. **剔除退货/取消单** —— `InvoiceNo` 以 `C` 开头；3. **只保留 Quantity > 0 且 UnitPrice > 0** —— 排除赠品与漏填；4. **派生字段** —— `TotalPrice = Quantity × UnitPrice`，并提取月份/小时/星期；5. **99 分位截断** —— 单价与单笔金额各截到 99 分位，避免极端批发单主导分层。> ⚠️ 保留率约 72%。截断会**连带删除**部分客户的全部交易（如只买过一件超贵商品的人），> 所以最终客户数是 4,284 而非数据集原始的 4,372 —— 这是预期结果，不是 bug。

In [ ]:
raw_rows = len(df)# ① 剔除无客户 IDdf = df.dropna(subset=['CustomerID'])# ② 剔除退货 / 取消订单df = df[~df['InvoiceNo'].astype(str).str.upper().str.startswith('C')]# ③ 只保留数量与单价为正df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]# ④ 派生字段df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])df['TotalPrice']  = (df['Quantity'] * df['UnitPrice']).round(2)df['OrderMonth']  = df['InvoiceDate'].dt.to_period('M').astype(str)df['Hour']        = df['InvoiceDate'].dt.hourdf['Weekday']     = df['InvoiceDate'].dt.dayofweek          # 0=周一 ... 6=周日# ⑤ 99 分位截断up_cap = df['UnitPrice'].quantile(0.99)tp_cap = df['TotalPrice'].quantile(0.99)df = df[(df['UnitPrice'] <= up_cap) & (df['TotalPrice'] <= tp_cap)]print(f'清洗前 {raw_rows:,} 行 → 清洗后 {len(df):,} 行，保留率 {len(df)/raw_rows:.1%}')print(f'时间跨度：{df.InvoiceDate.min()} ~ {df.InvoiceDate.max()}')print(f'用户数：{df.CustomerID.nunique():,}　订单数：{df.InvoiceNo.nunique():,}')print(f'总销售额：{df.TotalPrice.sum():,.0f} 英镑')df.to_csv('retail_clean.csv', index=False, encoding='utf-8-sig')

## 3. EDA：什么时候卖、卖什么、卖给谁三个问题对应三张主图，另外三张是补充视角。> **反直觉发现**：星期 × 小时热力图里**周末整行是空白的** —— 这家公司周末不处理订单。> 这个发现比图本身值钱，面试时讲出来能体现你"看图会想"。

In [ ]:
gmv_by_month = df.groupby('OrderMonth')['TotalPrice'].sum()gmv_by_hour  = df.groupby('Hour')['TotalPrice'].sum()pivot_wd = df.pivot_table(index='Weekday', columns='Hour',                          values='TotalPrice', aggfunc='sum')top_items = (df.groupby('Description')['Quantity'].sum()               .sort_values(ascending=False).head(10))top_country = (df.groupby('Country')                 .agg(orders=('InvoiceNo', 'nunique'),                      gmv=('TotalPrice', 'sum'))                 .sort_values('gmv', ascending=False).head(10))print('=== 月度 GMV ==='); print(gmv_by_month.round(0))print('\n=== 销售额最高的 5 个时段 ==='); print(gmv_by_hour.sort_values(ascending=False).head(5))print('\n=== 销量 TOP10 商品 ==='); print(top_items)print('\n=== GMV TOP10 国家 ==='); print(top_country.round(0))

In [ ]:
# 图1 月度 GMV 趋势 —— 9~11 月爬升，2011-12 骤降（该月只有 9 天数据，非真实下跌）fig, ax = plt.subplots(figsize=(11, 4))gmv_by_month.plot(kind='line', marker='o', color='#1f5fa9', ax=ax)ax.set_title('月度销售额趋势', fontsize=14)ax.set_ylabel('销售额（英镑）')plt.xticks(rotation=45); plt.tight_layout()plt.savefig('output/01_gmv_trend.png', dpi=150, bbox_inches='tight'); plt.show()# 图2 星期 × 小时 热力图 —— 周末整行空白plt.figure(figsize=(12, 4))sns.heatmap(pivot_wd, cmap='YlOrRd', linewidths=.3)plt.yticks(range(7), ['周一', '周二', '周三', '周四', '周五', '周六', '周日'], rotation=0)plt.xlabel('小时'); plt.title('星期 × 小时 销售热力图')plt.tight_layout(); plt.savefig('output/02_heatmap_weekday_hour.png', dpi=150, bbox_inches='tight'); plt.show()# 图3 销量 TOP10 商品fig, ax = plt.subplots(figsize=(9, 5))top_items.sort_values().plot(kind='barh', color='#1f5fa9', ax=ax)ax.set_title('销量 TOP10 商品', fontsize=14); ax.set_xlabel('销量（件）')plt.tight_layout(); plt.savefig('output/03_top_items.png', dpi=150, bbox_inches='tight'); plt.show()# 图4 GMV TOP10 国家fig, ax = plt.subplots(figsize=(9, 5))top_country['gmv'].sort_values().plot(kind='barh', color='#0b6b52', ax=ax)ax.set_title('销售额 TOP10 国家', fontsize=14); ax.set_xlabel('销售额（英镑）')plt.tight_layout(); plt.savefig('output/04_top_countries.png', dpi=150, bbox_inches='tight'); plt.show()# 图5 分时段销售额fig, ax = plt.subplots(figsize=(10, 4))gmv_by_hour.plot(kind='bar', color='#1f5fa9', ax=ax)ax.set_title('分时段销售额分布', fontsize=14)ax.set_xlabel('小时'); ax.set_ylabel('销售额（英镑）')plt.xticks(rotation=0); plt.tight_layout()plt.savefig('output/05_gmv_by_hour.png', dpi=150, bbox_inches='tight'); plt.show()# 图6 英国 vs 海外uk_share = (df.assign(is_uk=df['Country'] == 'United Kingdom')              .groupby('is_uk')['TotalPrice'].sum())plt.figure(figsize=(6, 6))plt.pie(uk_share, labels=['海外', '英国'], autopct='%.1f%%',        colors=['#f0a500', '#1f5fa9'], startangle=90)plt.title('英国 vs 海外销售额占比')plt.savefig('output/06_uk_share.png', dpi=150, bbox_inches='tight'); plt.show()

## 4. RFM 用户分层（最终版：价值与状态分离）### 方法迭代记录 ⭐这一节的方法是**改了三版**才定稿的，过程本身就是项目亮点：| 版本 | 做法 | 问题 || --- | --- | --- || v1 | R/F/M 各自 ≥ 均值即判为"高" | 高价值层占到 **30%**，区分度不足 || v2 | 加权综合分 `0.2R + 0.3F + 0.5M`，80 分位切层 | R 权重被 F/M 稀释，且 R 与 F/M 正相关 → "重要保持客户"只剩 **0.6%** || **v3（定稿）** | **价值由 F/M 决定，活跃状态由 R 独立判定** | 结构合理，通过聚类交叉验证 |**v3 的核心洞察**：R（近度）是**状态标签**，不是价值组成部分。"这个客户值多少钱"由消费行为（频次、金额）定义；"他现在活不活跃"由近度判断。两者揉进一个加权分数，流失信号会被自动掩盖。### 两个代码细节- `Recency` 基准日取 `最大日期 + 1 天`，否则最近下单的人 Recency = 0；- `Frequency` 有大量重复值（很多人只下过 1 单），`qcut` 前必须 `rank(method='first')`，否则报错。

In [ ]:
snapshot = df['InvoiceDate'].max() + dt.timedelta(days=1)rfm = df.groupby('CustomerID').agg(    Recency=('InvoiceDate', lambda x: (snapshot - x.max()).days),    Frequency=('InvoiceNo', 'nunique'),    Monetary=('TotalPrice', 'sum')).round(2)# 五分位打分：R 越小越好（标签反向），F/M 越大越好r_labels, fm_labels = [5, 4, 3, 2, 1], [1, 2, 3, 4, 5]rfm['R'] = pd.qcut(rfm['Recency'], q=5, labels=r_labels).astype(int)rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=fm_labels).astype(int)rfm['M'] = pd.qcut(rfm['Monetary'], q=5, labels=fm_labels).astype(int)# 价值分：只看 F、M（不含 R）rfm['Value'] = 0.4 * rfm['F'] + 0.6 * rfm['M']q80 = rfm['Value'].quantile(0.8)q50 = rfm['Value'].quantile(0.5)def seg3(row):    high = row['Value'] >= q80          # 前 20% = 高价值    mid  = row['Value'] >= q50    live = row['R'] >= 2                # 近期有回购 = 活跃（状态由 R 独立判定）    if high and live:  return '重要价值客户'    if high:           return '重要保持客户'   # 高价值但已流失 → 唤醒名单    if mid  and live:  return '潜力客户'    if mid:            return '一般保持客户'    if live:           return '一般发展客户'    return '一般挽留客户'rfm['Segment'] = rfm.apply(seg3, axis=1)summary = (rfm.groupby('Segment')              .agg(人数=('Recency', 'size'),                   人均R=('Recency', 'mean'),                   人均F=('Frequency', 'mean'),                   人均GMV=('Monetary', 'mean'),                   GMV=('Monetary', 'sum'))              .sort_values('GMV', ascending=False))summary['人数占比'] = (summary['人数'] / summary['人数'].sum()).map('{:.1%}'.format)summary['GMV占比']  = (summary['GMV']  / summary['GMV'].sum()).map('{:.1%}'.format)print(summary[['人数', '人数占比', '人均R', '人均F', '人均GMV', 'GMV占比']].round(1))rfm.to_csv('rfm.csv', encoding='utf-8-sig')

### 定稿结果解读| 人群 | 人数占比 | GMV 占比 | 运营含义 || --- | --- | --- | --- || 重要价值客户 | 22.8% | **69.5%** | 核心资产，重点维护 || 潜力客户 | 26.8% | 18.7% | **最值得投入的升级人群**（仍活跃，只是买得不够多） || 重要保持客户 | 0.3%（11 人） | — | 高价值客户流失率仅 1.1%，唤醒空间极小 || 一般挽留客户 | 17.0% | 低 | 人均 273 天未购，真流失，控制投入 |**结论**：这份数据的运营重点不是"唤醒流失"，而是"**推动潜力客户升级**"。

## 5. KMeans 交叉验证目的：验证上面的业务规则分层不是人为拍脑袋，而是**数据本身的结构**。三个判据：1. **轮廓系数选 k** —— 衡量几何分离度；2. **ARI（调整兰德指数）** —— 两套划分的一致性，0.3~0.6 属"结构吻合、边界不同"；3. **覆盖率** —— 高价值客户有多少落入聚类识别的高价值簇。> ⚠️ 诚实说明：两套方法用的是**同一组特征**（R/F/M），因此这不是严格意义上的独立验证，> 只能算"同一特征上两种划分逻辑的互相印证"。面试时主动点破这一点会加分。

In [ ]:
# 金额/频次严重右偏 → 先 log1p 再标准化feat = rfm[['Recency', 'Frequency', 'Monetary']].copy()X  = np.log1p(feat)Xs = StandardScaler().fit_transform(X)rows = []for k in range(2, 9):    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Xs)    rows.append({'k': k,                 'silhouette': round(silhouette_score(Xs, km.labels_), 4),                 'inertia': round(km.inertia_, 1)})k_df = pd.DataFrame(rows)print('=== 各 k 的聚类质量 ==='); print(k_df.to_string(index=False))fig, ax = plt.subplots(figsize=(7, 4))ax.plot(k_df['k'], k_df['silhouette'], marker='o', color='#1f5fa9')ax.set_xlabel('k'); ax.set_ylabel('轮廓系数'); ax.set_title('不同 k 下的聚类质量')plt.tight_layout(); plt.savefig('output/07_kmeans_k.png', dpi=150, bbox_inches='tight'); plt.show()# 轮廓系数 k=2 最优（数据天然分两大团），但业务需要更细粒度 → 取 k=6 与 RFM 分层对齐K = 6km = KMeans(n_clusters=K, random_state=42, n_init=10).fit(Xs)rfm['Cluster'] = km.labels_profile = (rfm.groupby('Cluster')              .agg(人数=('Recency', 'size'),                   人均R=('Recency', 'mean'),                   人均F=('Frequency', 'mean'),                   人均GMV=('Monetary', 'mean'),                   GMV=('Monetary', 'sum')))profile['人数占比'] = (profile['人数'] / profile['人数'].sum()).map('{:.1%}'.format)profile['GMV占比']  = (profile['GMV']  / profile['GMV'].sum()).map('{:.1%}'.format)print('\n=== 各簇画像 ==='); print(profile.round(1))# 按人均 GMV 降序给簇重命名，便于解读order = profile.sort_values('人均GMV', ascending=False).indexrfm['簇名'] = rfm['Cluster'].map({c: f'簇{i+1}' for i, c in enumerate(order)})ct     = pd.crosstab(rfm['Segment'], rfm['簇名'])ct_pct = ct.div(ct.sum(axis=1), axis=0).round(3)print('\n=== RFM 分层 × 聚类 行占比 ==='); print(ct_pct)plt.figure(figsize=(9, 5))sns.heatmap(ct_pct, annot=True, fmt='.2f', cmap='Blues')plt.title('RFM 分层 × KMeans 聚类 交叉占比')plt.tight_layout(); plt.savefig('output/08_crosstab.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# 定量一致性ari = adjusted_rand_score(rfm['Segment'], rfm['Cluster'])print(f'调整兰德指数 ARI = {ari:.3f}   （0.30~0.60 = 结构吻合、边界不同，属正常）')# 核心证据：高价值客户是否落入聚类识别的高价值簇top3 = profile.sort_values('人均GMV', ascending=False).head(3).indexcover = rfm['Cluster'].isin(top3)print('\n各人群落入"高价值簇"的比例：')# 用 groupby 取均值，避免 crosstab 布尔列在不同 pandas 版本下的索引歧义print(rfm.assign(在高价值簇=cover)         .groupby('Segment')['在高价值簇'].mean().round(3))# 簇价值排名 与 簇内高价值客户浓度 的相关性conc = pd.crosstab(rfm['Cluster'], rfm['Segment'], normalize='index')['重要价值客户']rho, p = spearmanr(profile.loc[conc.index, '人均GMV'], conc)print(f'\n簇人均GMV 与 重要价值客户浓度的 Spearman 相关 = {rho:.3f}')rfm.to_csv('rfm.csv', encoding='utf-8-sig')profile.reset_index().to_csv('cluster_profile.csv', index=False, encoding='utf-8-sig')

### 验证结果- **97.9%** 的重要价值客户落入 KMeans 识别的高价值簇（簇 1/2/3 合计 33.3% + 54.1% + 10.6%）；- **ARI = 0.408**，落在合理区间；- 簇人均 GMV 与簇内高价值客户浓度的 **Spearman ρ = 0.986**（几乎完全单调一致）；- 轮廓系数 **k=2 最优（0.435）** 但只说明数据天然分两大团，业务上选 **k=6**（0.316，质量可接受）。> 以上为 Notebook 全量复跑的实测值。若在别处看到 98.4% / ARI 0.468 等略有出入的数字，> 以仓库代码重新跑一遍的结果为准 —— 每次改动分层逻辑后都应重跑刷新。**一句话讲给面试官**：KMeans 把高价值层按行为模式又细分了几档（边界更碎），但量级判断完全一致 —— 这说明我的 RFM 分层抓住了数据的真实结构。

## 6. Cohort 同期群留存`CohortIndex = 下单月 - 首次购买月`，0 = 首购当月（恒为 100%，分析时剔除）。### 两个必须知道的数据陷阱1. **左截断**：数据从 2010-12-01 开始，此前已有的存量客户都被算成"12 月新客"，   导致第一个同期群留存虚高（次月 36.5%）。严谨做法是剔除它再平均。2. **右截断**：2011-11 之后进来的客户没有"次月"可观察，矩阵右下角必然是 NaN。

In [ ]:
df['OM'] = df['InvoiceDate'].dt.to_period('M')cohort = df.groupby('CustomerID')['OM'].min().rename('CohortMonth')df = df.merge(cohort, on='CustomerID')df['CohortIndex'] = (df['OM'] - df['CohortMonth']).apply(lambda x: x.n)size = df.groupby('CohortMonth')['CustomerID'].nunique().rename('NewUsers')active = (df.groupby(['CohortMonth', 'CohortIndex'])['CustomerID']            .nunique().rename('Active').reset_index()            .merge(size, on='CohortMonth'))active['Retention'] = (active['Active'] / active['NewUsers']).round(4)retention = active.pivot(index='CohortMonth', columns='CohortIndex', values='Retention')print('留存矩阵（%）：'); print((retention * 100).round(1))print(f'\n次月留存（全部同期群）   ：{retention[1].mean():.2%}')print(f'次月留存（剔除首个同期群）：{retention[1].iloc[1:].mean():.2%}')retention.reset_index().to_csv('retention.csv', index=False, encoding='utf-8-sig')# 长表：给 Power BI 矩阵视觉对象用（行=CohortMonth 列=CohortIndex 值=Retention）long = retention.stack().reset_index()long.columns = ['CohortMonth', 'CohortIndex', 'Retention']long.to_csv('retention_long.csv', index=False, encoding='utf-8-sig')

### 结果解读- **平均次月留存 ≈ 20.3%**（剔除首群约 18.9%）→ **首购后 30 天是核心流失窗口**；- 留存曲线**不是单调衰减**，在第 3、9~11 个月明显回升 —— 这是零售的季节性（圣诞采购季客户周期性回流），  与 App 类数据"单调下滑"不同，面试被问"留存为什么回升"时答"节假日效应"即可。

## 7. 复购率 / 客单价 / 国家 KPI看板第 1 页的四个 KPI 卡片就来自这一节。

In [ ]:
orders_per_user = df.groupby('CustomerID')['InvoiceNo'].nunique()print(f'复购率：{(orders_per_user >= 2).mean():.2%}')order_amount = df.groupby('InvoiceNo')['TotalPrice'].sum()print(f'客单价：{order_amount.mean():.2f} 英镑')print(f'件单价：{df.TotalPrice.mean():.2f} 英镑')country_kpi = (df.groupby('Country')                 .agg(客户数=('CustomerID', 'nunique'),                      订单数=('InvoiceNo', 'nunique'),                      GMV=('TotalPrice', 'sum')))country_kpi['客单价']  = (country_kpi['GMV'] / country_kpi['订单数']).round(2)country_kpi['人均GMV'] = (country_kpi['GMV'] / country_kpi['客户数']).round(2)print('\n国家 KPI TOP10：'); print(country_kpi.sort_values('GMV', ascending=False).head(10))

## 8.（可选）Apriori 购物篮分析⚠️ **必须限定国家**，否则商品×订单的布尔矩阵会撑爆内存。这里挑 France。需要 `pip install mlxtend`。

In [ ]:
# from mlxtend.frequent_patterns import apriori, association_rules## basket = (df[df['Country'] == 'France']#             .groupby(['InvoiceNo', 'Description'])['Quantity']#             .sum().unstack().fillna(0)#             .clip(0, 1).astype(bool))## freq  = apriori(basket, min_support=0.03, use_colnames=True)# rules = association_rules(freq, metric='lift', min_threshold=1.2)# print(rules.sort_values('lift', ascending=False)#            .head(20)[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

---## 总结与后续**已完成的闭环**：原始 Excel → 清洗 → EDA → RFM 分层 → 聚类交叉验证 → Cohort 留存 → Power BI 看板。**还能加的三件事**：1. **SQL 复算**：把 `retail_clean.csv` 导入 SQLite，用窗口函数复算月度环比、各国 Top3，验证与 Python 一致；2. **购物篮分析**：跑通上面的 Apriori，补上"商品关联推荐"这一维度；3. **预测**：用 BG/NBD 模型（ lifetimes 库）预测客户未来价值，比 RFM 更进一步。